# Oracle SQL/PLSQL Study + Interview Preparation Notebook

This notebook follows the playlist order from Video 1 through Video 87.

Source boundary for this notebook:
- Detailed notes are written only from transcripts that exist in the current workspace.
- The current workspace contains 64 unique playlist positions, not the full 87 transcript files.
- Missing playlist positions are preserved in sequence and marked explicitly instead of being fabricated.

Missing transcript positions in the current workspace:
1, 3, 4, 5, 6, 9, 10, 12, 13, 15, 18, 19, 28, 36, 39, 40, 42, 44, 50, 51, 53, 55, 58

How to use these notes:
- Study the concept explanation first.
- Pay attention to the restrictions, edge cases, and practical differences.
- Use the short interview-ready conclusions for revision before interviews.
- Where the transcript quality was noisy, only the strongest defensible technical points are retained.

## Video 1 - Transcript Not Available

The transcript for Video 1 is not present in the current workspace, so no technical notes are included for this slot.

## Video 2 - Procedure Vs Function

- This question is usually asked at fresher level, but the follow-up restrictions and edge cases make it useful even for intermediate interviews.
- The first and most important difference:
  - A function must return a value.
  - A procedure does not return a value through the RETURN clause; it sends data back through OUT or IN OUT parameters.
- A function returns only one value logically.
  - That one value can be scalar or composite.
  - For example, it may return a number, string, record-like value, or collection, but it is still one return object.
- A procedure can send multiple values back through multiple OUT parameters.

- Invocation style:
  - A function can be called from SQL, including a SELECT statement.
  - A procedure cannot be called directly from a SELECT statement.
- Typical mental model:
  - Functions are mainly for computation and deriving values.
  - Procedures are mainly for implementing business flow or data-processing steps.

- A key interview restriction:
  - If a function contains DML, Oracle does not allow it to be called from a normal SELECT statement.
  - The reason is that a query is expected to behave like a read-consistent operation and should not change database state inside that query execution path.
- Important exception:
  - If the function is declared as an autonomous transaction, then it may contain DML and still be callable from SQL.

- A common follow-up interview question:
  - Can a function contain DML?
  - Answer: Yes.
  - But if it does, you normally cannot call it from a plain SELECT statement unless you use an autonomous transaction workaround.

- RETURN behaves differently in functions and procedures:
  - In a function, RETURN returns the result value to the caller.
  - In a procedure, RETURN only exits the procedure early. It does not return a value the way a function does.

- Example patterns discussed:
  - A procedure that updates salary information and uses OUT parameters to pass values back.
  - An early-exit pattern in a procedure:
    - If the employee number is NULL, the procedure simply exits using RETURN.
  - A function that accepts a department number and returns the average salary for that department.
  - Such a function can be used in SQL because it behaves like a value-producing expression.

```sql
SELECT user_defined_function(...) FROM dual;
```

- Another useful interview edge case:
  - A function may compile even if an executable path does not return a value.
  - But when you actually call it, Oracle can raise an error such as function returned without value.
- Practical takeaway:
  - Compilation success does not guarantee correct return-path behavior.
  - Compiler warnings can help identify functions that do not reliably return a value on all paths.

- Version-specific point for experienced candidates:
  - From Oracle 12.1 onward, local procedures and functions can be defined inside a WITH clause and used as part of the SQL statement itself.
  - These are not permanent stored database objects; they exist only for that statement.

- Good interview summary:
  - Function:
    - Must return one logical value.
    - Can be used in SQL.
    - Best suited for computation.
    - DML inside it restricts SQL usage unless autonomous transaction is used.
  - Procedure:
    - Returns data through OUT or IN OUT parameters.
    - Cannot be called in SQL SELECT.
    - Best suited for process flow and business logic orchestration.
    - RETURN only exits early.

## Video 3 - Transcript Not Available

The transcript for Video 3 is not present in the current workspace, so no technical notes are included for this slot.

## Video 4 - Transcript Not Available

The transcript for Video 4 is not present in the current workspace, so no technical notes are included for this slot.

## Video 5 - Transcript Not Available

The transcript for Video 5 is not present in the current workspace, so no technical notes are included for this slot.

## Video 6 - Transcript Not Available

The transcript for Video 6 is not present in the current workspace, so no technical notes are included for this slot.

## Video 7 - What Is Dual Table in Oracle

- Basic answer for interview:
  - DUAL is a dummy table provided by Oracle.
  - It is created as part of Oracle installation.
  - It contains one column, DUMMY, and traditionally one row with value X.
- This is usually enough for fresher-level interviews.

- More complete answer:
  - DUAL is used when a SELECT statement must be syntactically complete, but you do not actually need data from a user table.
  - In Oracle, a SELECT statement normally needs a FROM clause.
  - DUAL is the standard one-row source used in those cases.

```sql
SELECT * FROM dual;
```

- Why it exists:
  - Sometimes you want Oracle to evaluate an expression, call a function, fetch a pseudocolumn, or compute a value without querying a business table.
  - DUAL gives you a legal one-row source for that purpose.

- Common use cases shown:

```sql
SELECT 1 + 2 FROM dual;
```

```sql
SELECT (p * n * r) / 100 FROM dual;
```

```sql
SELECT SQRT(100) FROM dual;
```

```sql
SELECT USER FROM dual;
```

```sql
SELECT SYSDATE FROM dual;
```

- It is also useful for calling built-in or user-defined functions that return a single scalar value.

```sql
SELECT some_function(...) FROM dual;
```

- Another example discussed was using string functions such as SUBSTR from DUAL when you only want computed output and not table data.

- Important practical interpretation:
  - In all such cases, DUAL is not the source of meaningful business data.
  - It is just there to complete the syntax of the SELECT.

- Sequence usage:
  - DUAL is commonly used to fetch sequence pseudocolumns such as NEXTVAL and CURRVAL.

```sql
SELECT sequence_name.NEXTVAL FROM dual;
```

- PL/SQL angle:
  - In PL/SQL, expressions like DECODE(...) are often wrapped in a SELECT ... FROM dual pattern so that the expression can be evaluated through SQL.

- Interview follow-up:
  - Can you create your own table similar to DUAL?
  - Yes. A user can create a one-row, one-column table and use it in a similar way.
  - But DUAL is optimized by Oracle, so it is the better and conventional choice.
- Another practical point:
  - DUAL is available to all users without needing custom object setup for this purpose.

- Advanced examples mentioned:
  - Generating multiplication-style output using ROWNUM and CONNECT BY.
  - Printing pyramid-like string patterns using SUBSTR, ROWNUM, and hierarchical query logic.
- The key learning from those examples is not the pattern itself, but the reason DUAL is involved:
  - No business table is needed.
  - The query is using SQL as a computation engine.

- Good interview summary:
  - DUAL is Oracle’s built-in dummy one-row table.
  - Use it whenever you need a valid FROM source only to evaluate an expression, call a function, fetch sequence values, or display system/user information.
  - It is lightweight, conventional, and optimized for exactly that purpose.

## Video 8 - What Is Trigger in Oracle

- Core interview definition:
  - A trigger is a PL/SQL code block or stored program unit that fires automatically when a specified database event occurs.
- The emphasis is on automatic execution.
  - A trigger is not manually invoked the way a procedure or function is.
  - The event causes the trigger to fire.

- Teaching analogy used:
  - A banking action such as deposit or withdrawal is the event.
  - SMS or email notifications are automatically triggered responses.
  - The same idea applies in Oracle:
    - Insert/update/delete or other database events occur.
    - Associated trigger logic runs automatically.

- Formal understanding:
  - Event happens.
  - Trigger associated with that event fires automatically.
  - The event can be DML, DDL, or a system event.

- Main trigger categories by event type:

- DML trigger:
  - Fires for data manipulation events such as:
    - INSERT
    - UPDATE
    - DELETE
  - These are typically defined on tables.
  - Typical use case: react whenever table data changes.

- DDL trigger:
  - Fires for data definition events such as:
    - CREATE
    - ALTER
    - DROP
    - TRUNCATE
    - GRANT
    - REVOKE
  - Typical use case: control or audit schema-level changes.

- System trigger:
  - Fires for system-level events such as:
    - LOGON
    - LOGOFF
    - database startup
    - database shutdown
  - Typical use case: auditing sessions, startup/shutdown housekeeping, cleanup tasks.

- Additional trigger types highlighted:

- INSTEAD OF trigger:
  - Usually written on a view.
  - Since a view may not directly support certain DML operations in the intended form, the trigger intercepts the operation and redirects the logic to underlying base tables.
  - This is why it is called INSTEAD OF: the trigger logic executes instead of the default attempted DML behavior on the view.
- Interview distinction:
  - A normal DML trigger is typically on a table.
  - An INSTEAD OF trigger is typically on a view.

- Compound trigger:
  - Conceptually related to DML trigger processing.
  - Lets you capture multiple timing-phase behaviors inside one trigger body rather than writing separate trigger units for each timing point.
  - Main advantage from an interview perspective: consolidated trigger logic for one DML event family.

- Important operational point:
  - You do not execute a trigger manually.
  - You cause its event, and Oracle fires it.

- Practical purposes of triggers discussed:

- Auditing data changes:
  - Capture what changed and when.
  - Potentially capture who changed it.
- Logging:
  - Maintain history of activity or transactions.
- Enforcing complex referential integrity:
  - Useful when business integrity rules are more complex than simple primary key or foreign key constraints.
- Transaction security:
  - Prevent certain operations such as deletes on protected tables.
  - A trigger can raise an exception and block the transaction.
- Data replication:
  - When a row is inserted in one table, trigger logic can copy related data into other tables.
- Preventing invalid data:
  - Example idea discussed: block suspicious or logically invalid values during insert/update.
- Auditing DDL operations:
  - Track who created, altered, or dropped objects.
  - Can also block operations such as dropping important tables.
- Auditing system activity:
  - Track session logon/logoff times.
  - Track how long sessions remain active.
- Startup/shutdown housekeeping:
  - Run cleanup or initialization tasks when the database starts or shuts down.

- Strong interview summary:
  - A trigger is event-driven PL/SQL that fires automatically.
  - Main trigger families are DML, DDL, and system triggers.
  - Important special types include INSTEAD OF and compound triggers.
  - Common uses are auditing, logging, validation, security enforcement, replication, and system/session handling.

## Video 9 - Transcript Not Available

The transcript for Video 9 is not present in the current workspace, so no technical notes are included for this slot.

## Video 10 - Transcript Not Available

The transcript for Video 10 is not present in the current workspace, so no technical notes are included for this slot.

## Video 11 - Difference Between REPLACE and TRANSLATE

- Both REPLACE and TRANSLATE are string manipulation functions.
- Both are used to alter characters or text in a source string.
- The main interview focus is not just syntax, but when to use one over the other.

- REPLACE:
  - Replaces a substring with another substring.
  - Works at whole-substring level, not character-by-character.
  - Typical signature:
    - source string
    - text to find
    - replacement text

```sql
SELECT REPLACE('welcome to Oracle class', 'Oracle', 'Python')
FROM dual;
```

- Result:
  - welcome to Python class

- Key behavior of REPLACE:
  - Every occurrence of the target substring is replaced.
  - The replacement text can be shorter, longer, or empty.
  - Length does not have to match the original substring.

```sql
SELECT REPLACE('welcome to Oracle class', 'Oracle', 'U')
FROM dual;
```

- Result:
  - welcome to U class

```sql
SELECT REPLACE('welcome to Oracle class', 'Oracle', '')
FROM dual;
```

- Result:
  - welcome to  class

- Practical interpretation:
  - Use REPLACE when you want to replace a full word, token, phrase, or exact substring pattern.

- TRANSLATE:
  - Also performs character substitution, but in a very different way.
  - It works character-by-character.
  - It does not replace a whole word with another whole word.
  - It maps each character in the second argument to the corresponding character in the third argument.

```sql
SELECT TRANSLATE('welcome to Oracle class', 'Oracle', '12345')
FROM dual;
```

- Conceptual mapping from the example:
  - O -> 1
  - r -> 2
  - a -> 3
  - c -> 4
  - l -> 5
  - The sixth character from Oracle has no matching replacement character because the third argument is shorter.
  - Characters without a mapped replacement are removed.

- This is the most important difference:
  - REPLACE substitutes substring-to-substring.
  - TRANSLATE substitutes character-to-character.

- Character-count rule in TRANSLATE:
  - If the source mapping string and replacement mapping string are the same length, each character has a direct mapped replacement.
  - If the replacement mapping string is shorter, the extra source characters are removed from the result.

```sql
SELECT TRANSLATE('abcdefghijkl', 'abcd', '1234')
FROM dual;
```

- Conceptual result:
  - a -> 1
  - b -> 2
  - c -> 3
  - d -> 4

```sql
SELECT TRANSLATE('abcdefghijkl', 'abcd', '123')
FROM dual;
```

- Conceptual result:
  - a -> 1
  - b -> 2
  - c -> 3
  - d has no target character, so it is removed

- Another key interview point:
  - In TRANSLATE, when a character from the second argument has no corresponding character in the third argument, that character is dropped from the final string.
  - This is a removal behavior, not a leave unchanged behavior for those mapped source characters.

- Null/empty-string behavior difference:
  - With REPLACE, if the third argument is empty or null-like in intent, the matched substring is removed.
  - With TRANSLATE, using a null or empty replacement argument results in a null output rather than selective replacement.
- This is a classic follow-up question because it sharply separates the two functions.

- Good scenario-based guidance:
  - Use REPLACE when you want to swap one word or substring for another.
  - Use TRANSLATE when you want multiple one-character substitutions in a single pass, such as cleanup, normalization, or character masking.

- Good interview summary:
  - REPLACE is substring-based.
  - TRANSLATE is character-mapping-based.
  - REPLACE can replace a whole word with another word.
  - TRANSLATE cannot do whole-word replacement; it maps each character independently.
  - In TRANSLATE, extra mapped source characters with no replacement are removed.
  - Null/empty replacement handling differs sharply between the two.

## Video 12 - Transcript Not Available

The transcript for Video 12 is not present in the current workspace, so no technical notes are included for this slot.

## Video 13 - Transcript Not Available

The transcript for Video 13 is not present in the current workspace, so no technical notes are included for this slot.

## Video 14 - Can We Use DML and DDL Statements Inside Function?

- Short answer:
  - Yes, a function can contain DML, and the broader discussion in the video also refers to DDL-related restrictions in function usage contexts.
- But the real interview value is in explaining where such a function can and cannot be called.

- Starting rule recalled from the earlier discussion:
  - If a function contains DML and you try to call it from a SELECT statement, Oracle rejects that pattern because a query should remain read-consistent and should not modify database state in that same query path.
- Workaround:
  - Make the function an autonomous transaction function if you need SQL-callable behavior despite internal DML.

- Main clarification in this video:
  - The restriction is really about calling such a function from SQL.
  - The same function may still be callable from PL/SQL when used as an expression rather than through a SELECT.

- Case 1: calling the function as a PL/SQL expression
  - This works.

```plsql
DECLARE
  v_result NUMBER;
BEGIN
  v_result := function_containing_dml(...);
  COMMIT; -- or ROLLBACK
END;
/
```

- Why it works:
  - The function is being invoked as part of PL/SQL expression evaluation.
  - It is not being executed through a SQL query.
- Important transaction point:
  - Because the function has modified data, the caller still needs to decide transaction outcome.
  - A COMMIT or ROLLBACK is required at the PL/SQL level unless transaction handling is built differently.

- Case 2: calling the same function through SELECT ... FROM dual inside a PL/SQL block
  - This still fails with the same DML-in-query style restriction.

```plsql
DECLARE
  v_result NUMBER;
BEGIN
  SELECT function_containing_dml(...)
    INTO v_result
    FROM dual;
END;
/
```

- Why it fails:
  - Even though the code appears inside PL/SQL, the actual invocation path is still SQL.
  - SELECT semantics take precedence.
  - Oracle still treats it as a query trying to execute a function that performs DML.

- Very important distinction for interviews:
  - Inside PL/SQL block is not the deciding factor.
  - Called as an expression versus called through a SELECT statement is the deciding factor.
- That means:
  - Allowed:
    - function with DML called directly in assignment/expression-style PL/SQL
  - Not allowed:
    - same function called from a SELECT, whether standalone or inside PL/SQL
  - Workaround:
    - autonomous transaction

- Good interview phrasing:
  - A function containing DML cannot normally be called from SQL because SQL statements are expected to remain read-consistent and not change database state through that function call.
  - However, the same function can be invoked from PL/SQL as part of an expression.
  - If the invocation happens through SELECT, the SQL restriction still applies even inside an anonymous block, procedure, or function.

## Video 15 - Transcript Not Available

The transcript for Video 15 is not present in the current workspace, so no technical notes are included for this slot.

## Video 16 - Difference Between View and Materialized View

- Base concept:
  - Both a view and a materialized view are named queries.
  - In both cases, you create an object name for a SELECT statement and then access the result through that object name.
- They can be built on:
  - one base table
  - multiple base tables
  - existing views
- So neither object is limited to a single underlying table.

- Core similarity:
  - Both expose the result of a query.
- Core difference:
  - A view stores query definition metadata.
  - A materialized view stores the query result physically.

- View behavior:
  - A view does not store its own result rows physically.
  - When you query a view, Oracle executes the underlying query at runtime and fetches data from the base table(s).
- Materialized view behavior:
  - A materialized view stores the result set in physical storage.
  - When you query it, Oracle reads from that stored result rather than recomputing from base tables each time.

- Reconstructed example from the video:

```sql
CREATE OR REPLACE VIEW emp_10 AS
SELECT *
FROM emp
WHERE deptno = 10;
```

```sql
CREATE MATERIALIZED VIEW emp_10mv AS
SELECT *
FROM emp
WHERE deptno = 10;
```

- Querying them:

```sql
SELECT * FROM emp_10;
```

```sql
SELECT * FROM emp_10mv;
```

- What happens behind the scenes:
  - emp_10 executes the underlying query on emp each time.
  - emp_10mv reads from its own stored copy of that query result.

- Data freshness behavior:
  - A view always reflects current base-table data immediately.
  - A materialized view can become stale because its stored copy is separate.
- This leads to the second major interview difference:
  - View data is effectively online/current.
  - Materialized view data is delayed/offline until refreshed.

- Example scenario from the video:
  - Create both the view and the materialized view on employee rows from department 10.
  - Delete or truncate the base-table rows.
  - Then query both objects.
- Result:
  - The view shows the current base-table state, so the rows disappear immediately.
  - The materialized view may still show old rows because its physical copy has not yet been refreshed.

- Refresh concept:
  - Materialized views need refresh to synchronize stored data with base-table changes.
  - The video points to the DBMS_MVIEW.REFRESH procedure for this purpose.

```sql
EXEC DBMS_MVIEW.REFRESH('EMP_10MV');
```

- After refresh:
  - The materialized view storage is synchronized with current base-table contents.

- Three primary interview differences emphasized:

- Physical storage:
  - View: no stored result set
  - Materialized view: stored result set exists physically

- Data source at query time:
  - View: reads from base query/base tables at runtime
  - Materialized view: reads from its own stored snapshot/result set

- Refresh requirement:
  - View: no refresh needed because it always derives current data
  - Materialized view: refresh required to stay in sync with source data

- Performance implication:
  - View can cost more at runtime for complex queries because it re-executes the underlying logic whenever queried.
  - Materialized view can improve read performance because data is precomputed and stored locally.
  - The tradeoff is refresh overhead and possible staleness.

- Good interview summary:
  - A view is a logical named query with no physical data storage of its own.
  - A materialized view is a named query whose result is physically stored.
  - View results always reflect current base-table data immediately.
  - Materialized view results reflect the last refresh state, not necessarily the latest base-table state.
  - Views need no refresh; materialized views do.

## Video 17 - Oracle Set Operators: UNION, UNION ALL, INTERSECT, MINUS

Set operators combine the result sets of two or more queries into one result set. They are useful when a problem is easier to solve as multiple separate queries and then merge the outputs.

A motivating pattern is: one query finds the employee with maximum salary, another finds the employee with minimum salary, and the final requirement is to return both rows in one result set.

```sql
select 'MAX_SAL' as type_label, ename, sal
from emp
where sal = (select max(sal) from emp)
union
select 'MIN_SAL' as type_label, ename, sal
from emp
where sal = (select min(sal) from emp);
```

Core rules for all set operators:
- The number of columns must match across each query block.
- The data types of corresponding columns must match.
- If either rule is violated, Oracle raises an error.

### UNION
UNION combines the results of both queries and removes duplicates.

### UNION ALL
UNION ALL combines the results of both queries and keeps duplicates.

### INTERSECT
INTERSECT returns only the rows common to both query results.

### MINUS
MINUS returns rows from the first query that do not appear in the second query.

Interview summary:
- UNION: combine results, remove duplicates, sort.
- UNION ALL: combine results, keep duplicates, usually faster.
- INTERSECT: return common rows.
- MINUS: return rows from first query excluding second-query rows.

## Video 18 - Transcript Not Available

The transcript for Video 18 is not present in the current workspace, so no technical notes are included for this slot.

## Video 19 - Transcript Not Available

The transcript for Video 19 is not present in the current workspace, so no technical notes are included for this slot.

## Video 20 - Oracle Index and Types of Index

This video focuses on two practical tasks:
- how to create common index types
- how to inspect existing index metadata before or after creating them

Key operational rule:
- Before creating a new index, first check whether a suitable index already exists. Unnecessary indexes add storage and DML maintenance overhead.

Types covered:
- B-tree index
- Bitmap index
- Function-based index
- Reverse key index
- Composite index

Representative examples:

```sql
create index idx_emp_empno on emp_t(empno);
```

```sql
create bitmap index idx_emp_deptno on emp_t(deptno);
```

```sql
create index idx_emp_upper_ename on emp_t(upper(ename));
```

```sql
create index idx_emp_sal_rev on emp_t(sal) reverse;
```

Useful metadata views:
- USER_INDEXES
- USER_IND_COLUMNS
- USER_IND_STATISTICS
- USER_IND_EXPRESSIONS

Interview takeaway:
- Indexes improve read performance.
- Indexes also slow down insert/update/delete because they must be maintained.

## Video 21 - B-tree Index in Oracle

A B-tree index is the standard Oracle index type and the default index answer unless another type is explicitly discussed.

Core ideas:
- B-tree stands for balanced tree.
- Oracle stores ordered key values plus rowid references.
- Oracle uses the rowid to fetch the table row efficiently after index lookup.

Use B-tree indexes on high-cardinality columns such as:
- employee ID
- account number
- registration number
- mail ID

Common scan types highlighted:
- index range scan
- index unique scan
- index full scan
- index fast full scan
- min/max optimization through the ordered index structure

Practical summary:
- Best for high-cardinality columns.
- Very good for selective lookups.
- Adds DML maintenance cost.

## Video 22 - Bitmap Index

Bitmap indexes are designed for low-cardinality columns.

Typical examples:
- gender
- yes/no status
- pass/fail result
- small department-code sets

Core ideas:
- Bitmap index stores key membership in bitmap form instead of standard ordered tree rows.
- Useful for filtering and counting in reporting-style workloads.
- Usually a poor choice for heavily updated OLTP tables.

Major caution from the video:
- Concurrent DML on bitmap-indexed columns can cause broader locking and blocking behavior.

Interview summary:
- Use bitmap indexes for low-cardinality columns and low-DML tables.
- Avoid them for highly concurrent transactional updates.

## Video 23 - Function-Based Index

A function-based index stores the result of a function or expression instead of the raw column value.

Why it exists:
- If queries frequently use predicates such as UPPER(col), LOWER(col), or another derived expression, a normal index on the raw column may not be usable.

Example:

```sql
create index idx_emp_upper_ename
on emp_t(upper(ename));
```

Then a query like this can use the index:

```sql
select *
from emp_t
where upper(ename) = 'SCOTT';
```

Important rule:
- The query expression must match the indexed expression closely enough for Oracle to use the function-based index.

## Video 24 - Reverse Key Index

A reverse key index is a B-tree variation where Oracle reverses the stored key values before placing them in the index.

Main purpose:
- reduce index block contention when many inserts or updates cluster around similar key ranges

Main advantage:
- distributes hot key patterns across index leaf blocks

Main drawback:
- not suitable for normal range scans

Interview summary:
- Reverse key index solves index block contention.
- It sacrifices normal range-scan usefulness.

## Video 25 - Index Summary: Choosing Types, Checking Usage, Monitoring

This video consolidates the index lessons.

Choice summary:
- B-tree: high cardinality
- Bitmap: low cardinality and low DML
- Function-based: predicates use functions/expressions
- Reverse key: reduce block contention

How to verify usage:
- EXPLAIN PLAN
- V$SQL and V$SQL_PLAN for executed statements

How to monitor usage:

```sql
alter index index_name monitoring usage;
```

```sql
alter index index_name nomonitoring usage;
```

Then review usage metadata such as DBA_OBJECT_USAGE.

Interview summary:
- Create indexes only when justified.
- Verify they are actually used.
- Balance read gains against DML overhead.

## Video 26 - Types of DML Triggers and Order of Trigger Execution

DML triggers are classified by:
- timing: BEFORE or AFTER
- event: INSERT, UPDATE, DELETE
- level: statement-level or row-level

That yields 12 common DML trigger combinations.

How to identify them:
- BEFORE or AFTER decides timing.
- INSERT/UPDATE/DELETE decides DML event.
- FOR EACH ROW means row-level.
- Absence of FOR EACH ROW means statement-level.

Execution order when both statement-level and row-level triggers exist for one DML event:
1. BEFORE statement-level trigger
2. BEFORE row-level trigger
3. actual DML action
4. AFTER row-level trigger
5. AFTER statement-level trigger

Important distinction:
- Statement-level triggers can fire even when zero rows are affected.
- Row-level triggers fire only for actual affected rows.

## Video 27 - SQL*Loader

SQL*Loader is Oracle’s command-line utility for loading data from external files into Oracle tables.

Key files involved:
- data file
- control file
- log file
- bad file
- discard file

The control file tells Oracle:
- which input file to read
- target table
- field separator rules
- optional filtering rules
- load mode such as truncate or append-style behavior

Representative structure:

```sql
options (skip=1)
load data
infile 'sample_data.txt'
discardfile 'sample_data.dsc'
truncate
into table employee_details
when deptno = '10'
fields terminated by ','
(
  ename,
  job,
  sal,
  deptno
)
```

Execution pattern:

```bash
sqlldr username/password control=control_file.ctl
```

Interview summary:
- SQL*Loader is used for bulk loading file data into Oracle tables.
- Bad rows go to the bad file.
- Rows filtered by WHEN conditions go to the discard file.
- The log file is the first place to inspect after execution.

## Video 28 - Transcript Not Available

The transcript for Video 28 is not present in the current workspace, so no technical notes are included for this slot.

## Video 29 - Instead-of Trigger

An instead-of trigger is written on a view, not on a base table.

Why it is needed:
- A simple view on one table can often be updated directly.
- A complex view built from multiple base tables often cannot be updated automatically by Oracle.
- An instead-of trigger intercepts DML on that view and replaces Oracle’s default handling with custom logic.

Pattern:

```sql
create or replace trigger trg_emp_dept_iot
instead of insert
on employee_department_view
begin
  -- custom routing logic into base tables
end;
```

Interview summary:
- Instead-of triggers are used mainly on views.
- They allow DML against views that Oracle cannot update automatically.

## Video 30 - What Is an Exception and How to Handle It in PL/SQL

An exception is an abnormal situation during normal execution of a PL/SQL program.

The video distinguishes:
- compile-time errors
- runtime errors

In PL/SQL discussion, runtime errors are usually what are meant by exceptions.

Typical PL/SQL block structure:

```sql
declare
  ...
begin
  ...
exception
  when ... then
    ...
end;
```

Examples covered conceptually:
- NO_DATA_FOUND
- TOO_MANY_ROWS
- ZERO_DIVIDE
- DUP_VAL_ON_INDEX

Why exception handling matters:
- convert technical errors into cleaner business messages
- prevent abrupt failures from surfacing raw Oracle errors directly to callers
- support logging and recovery logic

## Video 31 - Types of Exceptions

The video classifies exceptions into:
- predefined exceptions
- user-defined exceptions

Predefined exceptions are further grouped into:
- named exceptions
- unnamed exceptions

Named predefined exceptions mentioned:
- NO_DATA_FOUND
- DUP_VAL_ON_INDEX
- CURSOR_ALREADY_OPEN
- SUBSCRIPT_BEYOND_COUNT
- SUBSCRIPT_OUTSIDE_LIMIT
- ZERO_DIVIDE

Unnamed exceptions:
- have an error code and description
- do not have a convenient predefined PL/SQL name for direct handling
- are often caught with OTHERS unless explicitly mapped later

User-defined exceptions:
- are raised by application/business rules
- are created and raised by the developer

## Video 32 - PRAGMA EXCEPTION_INIT

PRAGMA EXCEPTION_INIT is used to associate a name with an otherwise unnamed Oracle exception.

Why it is needed:
- unnamed Oracle exceptions do not come with direct PL/SQL names
- without mapping, they are usually handled only through OTHERS

Pattern:

```sql
declare
  e_value_too_large exception;
  pragma exception_init(e_value_too_large, -1438);
begin
  ...
exception
  when e_value_too_large then
    ...
end;
```

Interview summary:
- PRAGMA EXCEPTION_INIT maps an Oracle error number to a user-declared exception name.

## Video 33 - RAISE_APPLICATION_ERROR

This video compares RAISE and RAISE_APPLICATION_ERROR.

RAISE:
- raises a user-defined exception inside PL/SQL flow

RAISE_APPLICATION_ERROR:
- raises an Oracle-style application error with a custom code and message

Pattern:

```sql
raise_application_error(-20001, 'Denominator cannot be zero');
```

Allowed range highlighted:
- -20001 to -20999

Interview summary:
- Use RAISE for internal PL/SQL exception flow.
- Use RAISE_APPLICATION_ERROR when you want a meaningful custom error returned to the caller.

## Video 34 - SQLCODE and SQLERRM

Every exception in PL/SQL has:
- an error code
- an error message

Oracle provides:
- SQLCODE for the code
- SQLERRM for the message

Example:

```sql
exception
  when zero_divide then
    dbms_output.put_line(sqlcode);
    dbms_output.put_line(sqlerrm);
end;
```

Practical use:
- logging
- diagnostics
- troubleshooting inside exception handlers

Interview summary:
- SQLCODE returns the numeric error code.
- SQLERRM returns the descriptive error message.

## Video 35 - What Is Subquery and Its Types

This subtitle file is too corrupted to support reliable detailed reconstruction.

Safe takeaway:
- The intended topic is subqueries and their types.
- The later cleaner videos in this series, especially Videos 37 and 38, provide the reliable usable material on correlated subqueries, simple subqueries, and inline views.

## Video 36 - Transcript Not Available

The transcript for Video 36 is not present in the current workspace, so no technical notes are included for this slot.

## Video 37 - Correlated Subquery

Broad classification:
- non-correlated subquery
- correlated subquery

Definition:
- A correlated subquery is a subquery whose inner query depends on the outer query.
- The inner query references outer-query values and therefore executes relative to each outer row.

Difference from a simple subquery:
- Simple subquery: inner query executes independently first.
- Correlated subquery: inner query depends on outer query and runs per outer row.

Common pattern:
- EXISTS / NOT EXISTS logic
- row-by-row filtering based on outer-row values

Representative example:

```sql
select e.*
from emp e
where e.sal > (
  select avg(e2.sal)
  from emp e2
  where e2.deptno = e.deptno
);
```

Interview summary:
- Correlated subquery depends on the outer query.
- It commonly appears with EXISTS and NOT EXISTS.

## Video 38 - Simple Subquery vs Correlated Subquery vs Inline View

Simple subquery:
- inner query is independent
- inner query executes first
- its result becomes input to the outer query

Correlated subquery:
- inner query depends on the outer query
- executes per outer row

Inline view:
- a subquery used as a derived table, often in the FROM clause

Representative inline-view pattern:

```sql
select e.*
from emp e,
     (
       select deptno, avg(sal) as avg_sal
       from emp
       group by deptno
     ) d
where e.deptno = d.deptno
  and e.sal > d.avg_sal;
```

Interview summary:
- Simple subquery: independent inner query.
- Correlated subquery: dependent inner query.
- Inline view: subquery used like a temporary table in SQL.

## Video 39 - Transcript Not Available

The transcript for Video 39 is not present in the current workspace, so no technical notes are included for this slot.

## Video 40 - Transcript Not Available

The transcript for Video 40 is not present in the current workspace, so no technical notes are included for this slot.

## Video 41 - IN vs ANY

Core comparison from the video:
- IN and = ANY are equivalent.

Example:

```sql
select *
from t
where c in (30, 50, 70);
```

Equivalent:

```sql
select *
from t
where c = any (30, 50, 70);
```

Rules to remember:
- > ANY means greater than the smallest value in the set.
- < ANY means less than the greatest value in the set.
- != ANY is syntactically possible but usually not useful in practical filtering.

Interview summary:
- IN = = ANY
- > ANY compares effectively against the minimum candidate boundary
- < ANY compares effectively against the maximum candidate boundary

## Video 42 - Transcript Not Available

The transcript for Video 42 is not present in the current workspace, so no technical notes are included for this slot.

## Video 43 - RANK and DENSE_RANK as Aggregate Function and Analytical Function

Ranking methodology explained:
- Highest value gets rank 1.
- Tied values receive the same rank.

Difference:
- RANK allows gaps after ties.
- DENSE_RANK does not allow gaps after ties.

Aggregate form:
- Used to find the rank of a supplied value within an ordered set.

Representative pattern:

```sql
select
  rank(1600) within group (order by sal desc) as sal_rank,
  dense_rank(1600) within group (order by sal desc) as sal_dense_rank
from emp;
```

Analytic form:
- Used to rank each returned row.

```sql
select
  empno,
  ename,
  sal,
  rank() over (order by sal desc) as sal_rank,
  dense_rank() over (order by sal desc) as sal_dense_rank
from emp;
```

Possible extension:
- partition by department for per-department ranking

## Video 44 - Transcript Not Available

The transcript for Video 44 is not present in the current workspace, so no technical notes are included for this slot.

## Video 45 - What is Cursor and What Are the Types of Cursor in Oracle

A cursor is the memory structure Oracle uses to process an SQL statement and its result set. In PL/SQL, cursors matter especially when a query can return multiple rows that need controlled processing.

Core idea:
- If a query returns exactly one row, SELECT INTO is often enough.
- If a query returns multiple rows, a cursor gives row-by-row access.

Types covered:
- implicit cursor
- explicit cursor

Interview summary:
- A cursor is a pointer to the SQL processing context area.
- Implicit cursors are Oracle-managed.
- Explicit cursors are developer-managed for multi-row control.

## Video 46 - What is Explicit Cursor

An explicit cursor is fully managed by the developer.

Lifecycle:
1. declare
2. open
3. fetch
4. close

Representative pattern:

```sql
DECLARE
  CURSOR c_emp IS
    SELECT name
    FROM employee;

  l_name employee.name%TYPE;
BEGIN
  OPEN c_emp;

  LOOP
    FETCH c_emp INTO l_name;
    EXIT WHEN c_emp%NOTFOUND;
    DBMS_OUTPUT.PUT_LINE(l_name);
  END LOOP;

  CLOSE c_emp;
END;
/
```

Why it matters:
- useful for multi-row queries
- useful when per-row logic is needed

## Video 47 - Explain Cursor Attributes

Cursor attributes discussed:
- %ISOPEN
- %FOUND
- %NOTFOUND
- %ROWCOUNT

Meaning:
- %ISOPEN: whether the cursor is open
- %FOUND: whether the last fetch returned a row
- %NOTFOUND: whether the last fetch did not return a row
- %ROWCOUNT: number of rows fetched so far, or affected rows for implicit SQL cases

Representative pattern:

```sql
DECLARE
  CURSOR c_emp IS
    SELECT name FROM employee;
  l_name employee.name%TYPE;
BEGIN
  IF NOT c_emp%ISOPEN THEN
    OPEN c_emp;
  END IF;

  LOOP
    FETCH c_emp INTO l_name;
    EXIT WHEN c_emp%NOTFOUND;
    DBMS_OUTPUT.PUT_LINE('Row ' || c_emp%ROWCOUNT || ': ' || l_name);
  END LOOP;

  CLOSE c_emp;
END;
/
```

## Video 48 - Explain Parameterized Cursor and For Cursor

Parameterized cursor:
- accepts parameters
- makes one cursor reusable for different filter values

Example shape:

```sql
DECLARE
  CURSOR c_emp (p_deptno NUMBER) IS
    SELECT name
    FROM employee
    WHERE department = p_deptno;

  l_name employee.name%TYPE;
BEGIN
  OPEN c_emp(10);

  LOOP
    FETCH c_emp INTO l_name;
    EXIT WHEN c_emp%NOTFOUND;
    DBMS_OUTPUT.PUT_LINE(l_name);
  END LOOP;

  CLOSE c_emp;
END;
/
```

Cursor-for-loop:
- Oracle automatically opens, fetches, exits, and closes

```sql
DECLARE
  CURSOR c_emp IS
    SELECT name
    FROM employee;
BEGIN
  FOR rec IN c_emp LOOP
    DBMS_OUTPUT.PUT_LINE(rec.name);
  END LOOP;
END;
/
```

## Video 49 - Explain REF Cursor, Strongly Typed REF Cursor, and Weakly Typed REF Cursor

REF cursor is a cursor variable.

Difference from normal explicit cursor:
- explicit cursor: fixed query at declaration time
- REF cursor: variable opened for a query later at runtime

Strongly typed REF cursor:
- fixed row structure
- better compile-time safety

Weakly typed REF cursor:
- flexible row structure
- fewer compile-time guarantees

Interview summary:
- REF cursor is useful for returning result sets from procedures/functions and for dynamic result handling.

## Video 50 - Transcript Not Available

The transcript for Video 50 is not present in the current workspace, so no technical notes are included for this slot.

## Video 51 - Transcript Not Available

The transcript for Video 51 is not present in the current workspace, so no technical notes are included for this slot.

## Video 52 - CURSOR vs REF CURSOR

Comparison:
- Normal cursor is static.
- REF cursor is dynamic.
- Normal cursor is tied to a query at declaration time.
- REF cursor is a variable opened for a query later.

Normal cursor use case:
- local fixed query processing

REF cursor use case:
- dynamic result handling
- returning a query result set to another caller or layer

## Video 53 - Transcript Not Available

The transcript for Video 53 is not present in the current workspace, so no technical notes are included for this slot.

## Video 54 - Compound Trigger

A compound trigger consolidates multiple DML trigger timing sections into one trigger body.

Typical timing sections inside one trigger:
- BEFORE STATEMENT
- BEFORE EACH ROW
- AFTER EACH ROW
- AFTER STATEMENT

Conceptual structure:

```sql
CREATE OR REPLACE TRIGGER trg_compound
FOR INSERT OR UPDATE OR DELETE ON employee
COMPOUND TRIGGER

  BEFORE STATEMENT IS
  BEGIN
    NULL;
  END BEFORE STATEMENT;

  BEFORE EACH ROW IS
  BEGIN
    NULL;
  END BEFORE EACH ROW;

  AFTER EACH ROW IS
  BEGIN
    NULL;
  END AFTER EACH ROW;

  AFTER STATEMENT IS
  BEGIN
    NULL;
  END AFTER STATEMENT;

END trg_compound;
/
```

Advantages:
- centralizes related trigger logic
- supports shared state across timing points
- useful in mutating-trigger-style redesigns

## Video 55 - Transcript Not Available

The transcript for Video 55 is not present in the current workspace, so no technical notes are included for this slot.

## Video 56 - What is Mutating Trigger

Mutating trigger error occurs when a row-level trigger tries to read from or modify the same table that is currently being changed by the triggering DML.

Representative anti-pattern:

```sql
CREATE OR REPLACE TRIGGER trg_emp
AFTER UPDATE OF salary ON employee
FOR EACH ROW
DECLARE
  l_cnt NUMBER;
BEGIN
  SELECT COUNT(*)
  INTO l_cnt
  FROM employee
  WHERE employee_id = :NEW.employee_id;
END;
/
```

Interview summary:
- Mutating trigger is a row-level trigger problem.
- It happens when the firing table is queried or modified while it is already in flux due to the triggering statement.

## Video 57 - How to Solve Mutating Trigger

The solution is redesign, not a different style of querying inside the same row-level trigger.

Common redesign ideas:
- move table-level logic to statement-level processing
- capture row-level details first, process them later
- use a compound trigger when both row-level capture and statement-level processing are needed

Interview summary:
- Avoid querying or modifying the firing table inside a row-level trigger.
- Use statement-level or compound-trigger patterns instead.

## Video 58 - Transcript Not Available

The transcript for Video 58 is not present in the current workspace, so no technical notes are included for this slot.

## Video 59 - SQL to Delete Duplicate Records

Classic interview problem:
- identify duplicate rows
- keep one row
- delete the remaining copies

Common Oracle patterns:

Using ROWID:

```sql
DELETE FROM employee_t e
WHERE ROWID NOT IN (
  SELECT MIN(ROWID)
  FROM employee_t
  GROUP BY empno, ename, deptno, job, salary
);
```

Using analytic function:

```sql
DELETE FROM employee_t
WHERE ROWID IN (
  SELECT rid
  FROM (
    SELECT ROWID rid,
           ROW_NUMBER() OVER (
             PARTITION BY empno, ename, deptno, job, salary
             ORDER BY ROWID
           ) rn
    FROM employee_t
  )
  WHERE rn > 1
);
```

Interview follow-up:
- define clearly what makes rows duplicates
- confirm whether one copy must be preserved

## Video 60 - Maximum Number of Triggers on Same Table

The most defensible takeaway is the DML-trigger classification count.

For common DML trigger combinations:
- 2 timings: BEFORE, AFTER
- 3 events: INSERT, UPDATE, DELETE
- 2 levels: statement-level, row-level

Total:

$$
2 \times 3 \times 2 = 12
$$

Interview takeaway:
- The commonly discussed DML trigger combinations on the same table total 12 categories.

## Video 61 - Creating Same Type of Trigger on Same Table

This video continues the trigger-count discussion and introduces the idea that multiple triggers can coexist on the same table.

Key points:
- Oracle can have multiple triggers associated with the same table.
- Timing, event type, and row-vs-statement distinctions still matter.
- If multiple related triggers exist, execution order becomes an interview concern.

Safe interview takeaway:
- Multiple same-table trigger scenarios are possible.
- If execution order is important, validate design carefully rather than assuming.

## Video 62 - Oracle PRAGMA AUTONOMOUS_TRANSACTION

Autonomous transaction creates an independent transaction scope inside PL/SQL.

Why it is useful:
- Sometimes logging or auditing should remain committed even if the caller rolls back.

Representative pattern:

```sql
CREATE OR REPLACE PROCEDURE log_message(p_text VARCHAR2) IS
  PRAGMA AUTONOMOUS_TRANSACTION;
BEGIN
  INSERT INTO log_table(message_text, created_on)
  VALUES (p_text, SYSDATE);

  COMMIT;
END;
/
```

Key points:
- independent from the caller’s transaction
- must explicitly COMMIT or ROLLBACK its own work
- commonly used for logging, auditing, and diagnostics

## Video 63 - PRAGMA AUTONOMOUS_TRANSACTION Real-Time Use Case

Main real-time use case:
- persistent logging even when the main transaction rolls back

Typical pattern:
- main business transaction runs
- error occurs
- autonomous procedure writes error log and commits it
- main transaction rolls back separately

Interview takeaway:
- Autonomous transaction is often used for error logging or audit logging that must survive rollback of the main transaction.

## Video 64 - Oracle Advantages of Packages

Advantages highlighted:
- modular programming
- encapsulation and private helper logic
- better organization and maintainability
- performance benefit because package state and code loading can help repeated calls after first reference

Interview summary:
- Packages group related procedures, functions, variables, cursors, and types.
- They support private members in the package body.
- They improve structure, maintainability, and often runtime behavior.

## Video 65 - Constraint with DEFERRABLE and NOVALIDATE

Strongest defensible ideas from the noisy transcript:

NOVALIDATE:
- applies the constraint to future DML
- does not immediately force validation of all existing rows
- useful when historical data is dirty but new data must comply

DEFERRABLE:
- constraint checking can be postponed until transaction end
- useful when intermediate transaction steps may temporarily violate the rule before final valid state is reached

Interview takeaway:
- ENABLE NOVALIDATE is a strong production answer when existing data is dirty but future data quality must improve.
- DEFERRABLE is useful when final transaction state matters more than temporary intermediate state.

## Video 66 - Oracle Constraint Related Interview Question

Main question:
- Does creating a primary key always create an index automatically?

Teaching answer:
- Yes, Oracle creates an index-backed enforcement path for the primary key.
- The transcript presents this through a create-table, add-primary-key, then inspect constraint and index metadata flow.

Important nuance:
- In practice, existing suitable indexing can influence the exact implementation path.

Representative example:

```sql
CREATE TABLE employee (
  empno NUMBER,
  ename VARCHAR2(100)
);

ALTER TABLE employee
  ADD CONSTRAINT pk_employee PRIMARY KEY (empno);
```

## Video 67 - How to Exclude Duplicate Records While Insertion | Error Log Table

Scenario:
- mixed-quality bulk load
- good rows should succeed
- bad rows, especially duplicate/constraint-violating rows, should be captured without stopping the entire batch

Strong Oracle answer:
- use DBMS_ERRLOG and LOG ERRORS

Pattern:

```sql
BEGIN
  DBMS_ERRLOG.CREATE_ERROR_LOG(dml_table_name => 'TARGET_TABLE');
END;
/

INSERT INTO target_table (...)
SELECT ...
FROM source_table
LOG ERRORS INTO err$_target_table
REJECT LIMIT UNLIMITED;
```

Why it is useful:
- good rows are inserted
- bad rows are redirected into the error table
- bulk load does not stop at the first bad row

## Video 68 - How to Exclude Duplicate Records While Insertion | Part 2

Continuation of the previous topic.

Practical comparison implied:
- ordinary insert with duplicate violation can fail the statement
- insert with LOG ERRORS allows good rows to succeed while bad rows are captured

Interview takeaway:
- error logging is a production-friendly bulk-load answer when duplicates or data-quality issues are expected.

## Video 69 - FORCE VIEW, WITH CHECK OPTION, WITH READ ONLY

FORCE VIEW:
- allows view creation even if underlying base tables do not yet exist

```sql
CREATE OR REPLACE FORCE VIEW student_view AS
SELECT *
FROM student;
```

WITH CHECK OPTION:
- DML through the view must still satisfy the view’s filter condition

```sql
CREATE OR REPLACE VIEW v_dept10 AS
SELECT *
FROM employee
WHERE department = 10
WITH CHECK OPTION;
```

WITH READ ONLY:
- makes the view query-only from the client perspective

```sql
CREATE OR REPLACE VIEW v_emp_ro AS
SELECT *
FROM employee
WITH READ ONLY;
```

## Video 70 - Oracle FOR vs FORALL | Advantages and Limitations of FORALL

FOR loop:
- row-by-row procedural iteration
- often causes repeated PL/SQL-to-SQL context switching when DML is executed inside the loop

FORALL:
- bulk-bind feature for DML
- sends many bind values efficiently to one DML statement

Representative idea:

```sql
FORALL i IN 1 .. l_ids.COUNT
  UPDATE employee
  SET salary = l_salaries(i)
  WHERE employee_id = l_ids(i);
```

Advantages of FORALL:
- better bulk DML performance
- fewer context switches

Limitations:
- intended for DML, not arbitrary procedural steps
- best when data already exists in collections

## Video 71 - Oracle MERGE Statement | Example

MERGE is used to synchronize target data with source data in one statement.

Use case:
- update target rows when matching source rows exist
- insert target rows when they do not exist yet

Representative syntax:

```sql
MERGE INTO target_table t
USING source_table s
ON (t.employee_id = s.employee_id)
WHEN MATCHED THEN
  UPDATE SET t.salary = s.salary
WHEN NOT MATCHED THEN
  INSERT (employee_id, employee_name, salary)
  VALUES (s.employee_id, s.employee_name, s.salary);
```

Interview takeaway:
- MERGE is the standard set-based upsert/synchronization statement.

## Video 72 - Oracle MERGE Statement Error Handling

Key defensible points:
- MERGE is powerful, but bad source data or ambiguous matching can cause statement failure.
- The ON clause must be designed carefully.
- Duplicate or ambiguous source-to-target matching is a common MERGE risk.

Interview follow-up:
- What if multiple source rows match the same target row?
- Why is ON-clause design critical?

## Video 73 - View Related Question | Add/Drop Columns in Base Table

Classic view interview question.

Main conclusion:
- If a view is created with SELECT *, the asterisk is resolved to the base-table column list at creation time.
- Adding a new column to the base table does not automatically expose that new column through the existing view.
- Dropping a column used by the view can invalidate the view.

Interview takeaway:
- SELECT * inside a view is not dynamically re-expanded later.

## Video 74 - View Related Question Part 2 | Add/Drop Columns in Base Table

Continuation of Video 73.

Reinforced conclusions:
- base-table structural changes can invalidate dependent views
- dependency analysis should happen before DDL
- if new columns must appear in the view, recreate the view explicitly
- if a dropped column invalidates the view, repair the definition and recompile it

## Video 75 - Oracle Table Function

A table function is a PL/SQL function that returns a collection, and Oracle lets SQL treat that collection like a table by using the TABLE(...) operator.

Core flow:
1. define row shape
2. define collection type
3. populate collection in a function
4. return the collection
5. query it through TABLE(function_call)

Conceptual structure:

```sql
CREATE TYPE row_type AS OBJECT (...);
CREATE TYPE row_table_type AS TABLE OF row_type;

CREATE OR REPLACE FUNCTION fn_name(...)
  RETURN row_table_type
IS
  l_data row_table_type := row_table_type();
BEGIN
  -- populate l_data
  RETURN l_data;
END;
/

SELECT *
FROM TABLE(fn_name(...));
```

## Video 76 - Oracle Pipelined Table Function

A pipelined table function returns rows incrementally instead of building the complete collection first.

Normal table function:
- builds the whole collection first
- returns the collection at the end

Pipelined table function:
- emits rows one by one using PIPE ROW
- caller can start consuming rows earlier

Conceptual structure:

```sql
CREATE OR REPLACE FUNCTION fn_name(...)
  RETURN row_table_type PIPELINED
IS
BEGIN
  PIPE ROW(row_type(...));
  RETURN;
END;
/

SELECT *
FROM TABLE(fn_name(...));
```

## Video 77 - Oracle Pipelined Table Function Vs Table Function

Comparison summary:
- table function returns the complete collection after building it fully
- pipelined table function streams rows one by one
- pipelined version is usually better for larger result sets from memory and response-time perspectives

## Video 78 - What Is Data Integrity and Constraints | Types of Constraints

Data integrity means the data stored in the database must obey the application’s business rules.

Examples from the video:
- register number should be unique
- gender should allow only approved values
- joining date should not be in the future

Constraints are one of the main Oracle mechanisms used to enforce those business rules.

Types listed:
- NOT NULL
- UNIQUE
- PRIMARY KEY
- FOREIGN KEY
- CHECK
- REF

Strong revision line:
- Data integrity is the business requirement.
- Constraints are one of the database-level enforcement tools.

## Video 79 - Oracle NOT NULL Constraint

NOT NULL prevents a column from containing NULL.

Use it when a column is mandatory for the business process.

Example:

```sql
CREATE TABLE emp_demo (
  emp_id   NUMBER NOT NULL,
  emp_name VARCHAR2(100) NOT NULL
);
```

Short comparison:
- NOT NULL means a value must exist.
- UNIQUE means values must not repeat.

## Video 80 - UNIQUE Constraint in Oracle

UNIQUE enforces that values in a column, or a combination of columns, must be distinct.

Important behavior:
- duplicate non-null values are not allowed
- nulls are still allowed

Single-column example:

```sql
CREATE TABLE users_demo (
  user_id   NUMBER,
  username  VARCHAR2(50) UNIQUE
);
```

Composite example:

```sql
CREATE TABLE registration_demo (
  student_id NUMBER,
  course_id  NUMBER,
  CONSTRAINT uq_student_course UNIQUE (student_id, course_id)
);
```

## Video 81 - Oracle PRIMARY KEY Constraint

Primary key characteristics emphasized:
1. values must be unique
2. nulls are not allowed
3. Oracle automatically creates index-backed enforcement for it

Example:

```sql
CREATE TABLE employee_demo (
  emp_id   NUMBER PRIMARY KEY,
  emp_name VARCHAR2(100)
);
```

Best revision line:
- Primary key = UNIQUE + NOT NULL + row-identifier role.

## Video 82 - Oracle FOREIGN KEY Constraint

A foreign key creates a parent-child relationship between tables.

Definition:
- child table contains the foreign key
- parent table contains the referenced primary key or unique key
- child rows must reference existing parent values

Example:

```sql
CREATE TABLE customer_demo (
  customer_id NUMBER PRIMARY KEY,
  customer_name VARCHAR2(100)
);

CREATE TABLE order_demo (
  order_id     NUMBER PRIMARY KEY,
  customer_id  NUMBER,
  CONSTRAINT fk_order_customer
    FOREIGN KEY (customer_id)
    REFERENCES customer_demo (customer_id)
);
```

Interview summary:
- foreign key enforces referential integrity and prevents orphan records.

## Video 83 - Oracle CHECK Constraint

CHECK enforces a boolean condition that every row must satisfy.

Examples:

```sql
CREATE TABLE person_demo (
  age NUMBER,
  CONSTRAINT chk_age_positive CHECK (age > 0)
);
```

```sql
CREATE TABLE employee_demo (
  salary NUMBER,
  gender CHAR(1),
  start_date DATE,
  end_date DATE,
  CONSTRAINT chk_salary_positive CHECK (salary > 0),
  CONSTRAINT chk_gender CHECK (gender IN ('M', 'F')),
  CONSTRAINT chk_dates CHECK (start_date <= end_date)
);
```

Interview summary:
- Use CHECK when the rule can be expressed as a row-level condition.

## Video 84 - Oracle REF Constraint

Main distinction from the transcript:
- REF constraint is not the same as a normal foreign key.
- REF belongs to Oracle object-relational design.
- Foreign key belongs to standard relational-table design.

Safe interview answer:
- REF constraint is used in Oracle object-table/object-relational contexts, while foreign key is used in ordinary relational parent-child table design.

## Video 85 - DML DDL TCL Commands in Procedure, Function, and Trigger

Key teaching points captured clearly:

DDL inside PL/SQL body:
- do not write it as a normal static DDL line inside the block body
- use dynamic SQL instead

Example:

```sql
CREATE OR REPLACE PROCEDURE p1 IS
BEGIN
  EXECUTE IMMEDIATE 'CREATE TABLE test_tab1 (...)';
END;
/
```

Function with DML:
- can be created
- can be called from PL/SQL expression context
- cannot normally be called from a SQL SELECT statement if it performs DML

Autonomous transaction workaround:

```sql
CREATE OR REPLACE FUNCTION f1
  RETURN NUMBER
IS
  PRAGMA AUTONOMOUS_TRANSACTION;
BEGIN
  INSERT INTO some_table ...;
  COMMIT;
  RETURN 1;
END;
/
```

Important rule:
- autonomous transaction blocks should explicitly COMMIT or ROLLBACK

## Video 86 - External Table in Oracle Database

An external table lets Oracle read data stored outside the database as if it were querying a table.

Important distinction:
- table metadata is in Oracle
- row data remains in external files

Typical steps:
1. create directory object
2. create external table using ORGANIZATION EXTERNAL
3. query it with SELECT

Conceptual structure:

```sql
CREATE OR REPLACE DIRECTORY external_directory AS 'D:\temp\data';
```

```sql
CREATE TABLE ext_emp (
  empno   NUMBER,
  ename   VARCHAR2(100),
  hiredate DATE,
  deptno  NUMBER
)
ORGANIZATION EXTERNAL (
  TYPE ORACLE_LOADER
  DEFAULT DIRECTORY external_directory
  ACCESS PARAMETERS (
    -- file format metadata
  )
  LOCATION ('employee_data.txt')
);
```

```sql
SELECT *
FROM ext_emp;
```

Interview summary:
- External tables expose file data through SQL.
- Data remains outside the database.

## Video 87 - External Table Related Questions

Key practical interview answers:

Can DML be done on an external table?
- No. The video demonstrates external tables as read-only here.

Can TRUNCATE be used?
- No. The transcript shows it as unsupported.

Can ALTER be used?
- Some metadata-level changes are possible, such as adding a column definition.

How does Oracle map file data?
- primarily by position and datatype compatibility

What if mapping or datatype interpretation fails?
- rejected rows go to the bad file
- diagnostic details go to the log file

Strong revision summary:
- External tables are read-only SQL access paths over external files.
- Column order, datatype compatibility, bad files, and log files are critical for troubleshooting.